# FX Graph Mode Post-Training Quantization 

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


from utils.preprocessing import prepare_dataloaders, load_mobilenetv2_model
from utils.metrics import compute_top_k_accuracy, benchmark_model, print_model_size
from utils.post_training import quantize_model

In [ ]:
# Set up warnings
import warnings
warnings.filterwarnings(
    action='ignore',
    category=DeprecationWarning,
    module=r'.*'
)
warnings.filterwarnings(
    action='default',
    module=r'torch.ao.quantization'
)

# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

### Hyperparameters

In [ ]:
# Paths for the dataset and model weights
data_dir = '../../../../EdgeComputingGroup/model-compression/data/skin-lesions/download'  # Update with your dataset path
model_weights_path = '../../mobilenet_v2_best_model.pth'  # Update with your saved model weights

train_batch_size = 32
eval_batch_size = 32

# Device configuration - quantization is often done on CPU.
device = torch.device("cpu")

### Helper Functions

In [ ]:
def load_quantized_model(model_path: str, device: torch.device = torch.device("cpu")) -> torch.jit.ScriptModule:
    """
    Loads a quantized TorchScript model from the specified file.

    Parameters:
        model_path (str): Path to the saved TorchScript model (.pt file).
        device (torch.device): The device on which to load the model. Defaults to CPU.
    
    Returns:
        torch.jit.ScriptModule: The loaded quantized model.
    """
    # Load the TorchScript model from disk, mapping it to the specified device.
    model = torch.jit.load(model_path, map_location=device)
    # Set the model to evaluation mode (important for inference)
    model.eval()
    return model


def evaluate_model(model: torch.nn.Module, data_loader: DataLoader, criterion: nn.Module) -> None:
    """
    Evaluates the model on the evaluation data and prints accuracy.
    
    Parameters:
        model (nn.Module): The model to evaluate.
        data_loader (DataLoader): DataLoader for the evaluation dataset.
        criterion (nn.Module): Loss function.
    """
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_samples = 0
    with torch.no_grad():
        for images, targets in data_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * images.size(0)
            acc1 = compute_top_k_accuracy(outputs, targets, topk=(1,))[0]
            total_top1 += acc1 * images.size(0) / 100.0  # converting percent back to count
            total_samples += images.size(0)
    
    avg_loss = total_loss / total_samples
    avg_acc = total_top1 / total_samples * 100.0
    print(f"Evaluation Loss: {avg_loss:.4f}, Top-1 Accuracy: {avg_acc:.2f}%")


### Post-Training Quantization

### Main

In [9]:
"""
Main function to load a MobileNetV2 skin lesion classifier, perform FX quantization,
and evaluate the model before and after quantization.
"""
# Prepare data loaders
train_loader, eval_loader = prepare_dataloaders(data_dir, train_batch_size)

num_class = len(train_loader.dataset.classes)

# Load the pre-trained (fine-tuned) MobileNetV2 model
original_model = load_mobilenetv2_model(model_weights_path, num_class, de)

# Choose an example input from the training data (we only need the image, not the label)
example_input = next(iter(train_loader))[0].to(device)

# Perform quantization
quantized_model = quantize_model(original_model, example_input, eval_loader)

# Optionally, save the quantized model using TorchScript for later deployment
quantized_model_scripted = torch.jit.script(quantized_model)
save_path = os.path.join(os.path.dirname(model_weights_path), "mobilenetv2_quantized.pt")
torch.jit.save(quantized_model_scripted, save_path)
print(f"Quantized model saved to: {save_path}")

NameError: name 'de' is not defined

### Benchmarks

In [ ]:

# original_model = load_mobilenetv2_model(model_weights_path, num_class)
# quantized_model = load_quantized_model("mobilenetv2_quantized.pt")

# # Benchmark the quantized model.
# metrics = benchmark_model(quantized_model, eval_loader, nn.CrossEntropyLoss())

# Print the size of the float model
print("Float model size:")
print_model_size(original_model)

# Print the size of the quantized model
print("Quantized model size:")
print_model_size(quantized_model)

# # Evaluate both models using a suitable loss function (e.g., cross entropy)
# criterion = nn.CrossEntropyLoss() S
# print("Evaluation of quantized model:")
# evaluate_model(quantized_model, eval_loader, criterion)

### Quantization Aware Training

In [ ]:
def train_qat_model(model: nn.Module, train_loader, device: torch.device, epochs: int = 10, lr: float = 1e-4) -> nn.Module:
    """
    Trains the QAT model using a standard training loop with cross entropy loss.

    Parameters:
        model (nn.Module): The QAT-prepared model.
        train_loader: DataLoader for the training dataset.
        device (torch.device): Device to perform training on.
        epochs (int): Number of training epochs.
        lr (float): Learning rate.
    
    Returns:
        nn.Module: The trained QAT model.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()  # Ensure model is in training mode
        running_loss = 0.0
        
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
    
    return model

In [ ]:
# Prepare data loaders
train_loader, eval_loader = prepare_skin_lesion_dataloaders(data_dir)

num_class = len(train_loader.dataset.classes)

# Load the pre-trained (fine-tuned) MobileNetV2 model
model = load_mobilenetv2_model(model_weights_path, num_class)


# Use the custom qconfig in your FX quantization flow
custom_qconfig = get_custom_qconfig()
# qconfig_mapping = torch.ao.quantization.QConfigMapping().set_global(custom_qconfig)
model.qconfig = custom_qconfig

model.train()  # QAT requires training mode
qat_model = prepare_qat(model, inplace=True)

qat_model = train_qat_model(qat_model, train_loader, device, epochs=2, lr=1e-4)

# optimizer = torch.optim.SGD(original_model.parameters(), lr = 0.0001)
# # The old 'fbgemm' is still available but 'x86' is the recommended default.
# original_model.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

### Debug

In [ ]:
import matplotlib.pyplot as plt

def debug_quantized_model(original_model: nn.Module, quantized_model: nn.Module) -> list[int]:
    """
    Comparing Weights Using the Numeric Suite.
    
    Parameters:
        original_model (nn.Module): The original model to comapre to.
        quantized_model (nn.Module): The quantized model to inspect.
    """
    print("Debugging quantized model parameters:")

    # Extract weight pairs between the original (float) and quantized models.
    # Here we label the weights from the float model as 'float' and from the quantized model as 'quantized'.
    weight_comparison = ns.extract_weights('float', original_model, 'quantized', quantized_model)

    # Extend the comparison dictionary by computing the SQNR (Signal-to-Quantization-Noise Ratio)
    ns.extend_logger_results_with_comparison(
        weight_comparison, 'float', 'quantized', torch.ao.ns.fx.utils.compute_sqnr, 'sqnr'
    )

    sqnr_list = []

    # Now, print the SQNR values for each weight pair for inspection.
    print("Weight Comparison (SQNR):")
    for name, comp in weight_comparison.items():
        # Each 'comp' contains entries for both sides and the computed SQNR.
        sqnr = comp['weight']['quantized'][0].get('sqnr', 'N/A')
        sqnr_list.append(sqnr[0])
        print(f"{name}: SQNR = {sqnr}")

    return sqnr_list

def plot_sqnr(xdata, ydata, xlabel, ylabel, title):
    # Create the plot
    fig = plt.figure(figsize=(10, 5))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    
    # Use plt.gca() to get the current axes (which uses data coordinates)
    ax = plt.gca()

    # Plot xdata vs ydata
    ax.plot(xdata, ydata)

    print(min(ydata))

    # Set the x and y axis limits to their respective min and max values
    ax.set_xlim([min(xdata)-1, max(xdata)+1])
    ax.set_ylim([min(ydata)-1, max(ydata)+1])

    # Set the x and y ticks to increment by 1
    ax.set_xticks(range(min(xdata), max(xdata), 2))  # Set x ticks from min to max with step 2
    # ax.set_yticks(range(min(ydata), max(ydata) + 1, 5))  # Set y ticks from min to max with step 1

    plt.tight_layout()

    # Show the plot
    plt.show()


# original_model = load_mobilenetv2_model(model_weights_path, num_class)
# quantized_model = load_quantized_model("mobilenetv2_quantized.pt")

# Example usage:
# Assuming quantized_model is your quantized model instance
ydata = debug_quantized_model(original_model, quantized_model)
xdata = list(range(len(ydata)))
print(xdata)
plot_sqnr(xdata, ydata, "idx", "sqnr", "sqnr vs idx")